# Notebook 2: Keyword baselines (Random, TF-IDF, BM25)

This notebook produces the **first rows of the results table**. For every job in the test split, each method ranks that job's candidate resumes. We then score the ranking against the dataset labels.

**Metrics** (averaged over jobs; 95% confidence intervals by bootstrap over jobs):
- **NDCG@5 / NDCG@10**: graded (Good = 2, Potential = 1, No = 0). Rewards putting the best candidates at the very top.
- **P@5**: share of the top 5 that are a *Good Fit*.
- **MRR**: how high the first *Good Fit* appears (1 = first place, 0.5 = second place, …).
- **Latency**: milliseconds to rank one job's candidates.

These methods need no training, so the resume overlap between train and test (see Notebook 1) does not affect them.

**Run with:** Runtime → Run all. It needs the files Notebook 1 saved to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install rank-bm25

import re, time, json, os
import numpy as np
import pandas as pd

DATA = '/content/drive/MyDrive/resume-job-matching/data'
RESULTS = '/content/drive/MyDrive/resume-job-matching/results'
os.makedirs(RESULTS, exist_ok=True)
SEED = 42

train_df = pd.read_parquet(f'{DATA}/train.parquet')
val_df = pd.read_parquet(f'{DATA}/val.parquet')
test_df = pd.read_parquet(f'{DATA}/test.parquet')
print(len(train_df), len(val_df), len(test_df))

## 1. Metrics
The same functions are used in every later notebook, so all methods are scored identically.

In [ ]:
def dcg(rels):
    rels = np.asarray(rels, dtype=float)
    return np.sum((2 ** rels - 1) / np.log2(np.arange(2, len(rels) + 2)))

def ndcg_at_k(ranked_rels, k):
    ideal = dcg(sorted(ranked_rels, reverse=True)[:k])
    return dcg(ranked_rels[:k]) / ideal if ideal > 0 else 0.0

def precision_at_k(ranked_rels, k, min_rel=2):
    return np.mean([r >= min_rel for r in ranked_rels[:k]])

def mrr(ranked_rels, min_rel=2):
    for i, r in enumerate(ranked_rels):
        if r >= min_rel:
            return 1.0 / (i + 1)
    return 0.0

def score_query(ranked_rels):
    return {
        'NDCG@5': ndcg_at_k(ranked_rels, 5),
        'NDCG@10': ndcg_at_k(ranked_rels, 10),
        'P@5': precision_at_k(ranked_rels, 5),
        'MRR': mrr(ranked_rels),
    }

def bootstrap_ci(values, n=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    values = np.asarray(values)
    means = [rng.choice(values, len(values), replace=True).mean() for _ in range(n)]
    return np.percentile(means, [2.5, 97.5])

# Quick self-test of the metrics
assert ndcg_at_k([2, 1, 0], 3) == 1.0
assert mrr([0, 0, 2]) == 1 / 3
assert precision_at_k([2, 0, 2, 0, 0], 5) == 0.4
print('metrics OK')

## 2. Evaluation loop
`score_fn(job_text, candidate_texts)` returns one score per candidate (higher = better match). Ties are broken at random with a fixed seed, so a method can't look good by luck of the input order.

In [ ]:
def evaluate(method_name, score_fn, df, split_name='test'):
    rng = np.random.default_rng(SEED)
    rows, latencies = [], []
    usable = df[df.eval_usable] if 'eval_usable' in df else df
    for job_id, group in usable.groupby('job_id'):
        job_text = group.job_text.iloc[0]
        t0 = time.perf_counter()
        scores = np.asarray(score_fn(job_text, group.resume_text.tolist()), dtype=float)
        latencies.append((time.perf_counter() - t0) * 1000)
        tiebreak = rng.random(len(scores))
        order = np.lexsort((tiebreak, -scores))  # sort by score desc, then random
        ranked_rels = group.relevance.to_numpy()[order].tolist()
        rows.append({'method': method_name, 'split': split_name, 'job_id': job_id,
                     'candidates': len(group), **score_query(ranked_rels)})
    per_query = pd.DataFrame(rows)
    summary = {'method': method_name, 'split': split_name, 'jobs': len(per_query)}
    for m in ['NDCG@5', 'NDCG@10', 'P@5', 'MRR']:
        lo, hi = bootstrap_ci(per_query[m])
        summary[m] = round(per_query[m].mean(), 4)
        summary[m + ' CI'] = f'[{lo:.3f}, {hi:.3f}]'
    summary['ms/job (median)'] = round(float(np.median(latencies)), 2)
    return summary, per_query

## 3. Random baseline
Tells us what "no skill at all" scores on this data. Every real method must clearly beat it.

In [ ]:
def random_scores(job_text, candidates, _rng=np.random.default_rng(SEED)):
    return _rng.random(len(candidates))

## 4. TF-IDF + cosine similarity
The vocabulary and IDF weights are learned from the **train** split only (jobs and resumes). Unigrams and bigrams, English stop words removed, sublinear TF.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train_corpus = pd.concat([train_df.job_text.drop_duplicates(), train_df.resume_text.drop_duplicates()])
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2,
                        sublinear_tf=True, max_features=100_000)
tfidf.fit(train_corpus)
print('TF-IDF vocabulary size:', len(tfidf.vocabulary_))

def tfidf_scores(job_text, candidates):
    job_vec = tfidf.transform([job_text])
    cand_vecs = tfidf.transform(candidates)
    return cosine_similarity(job_vec, cand_vecs).ravel()

## 5. BM25 (Okapi)
The standard keyword-search ranking function. The job description is the query. Document statistics (IDF, average length) come from all unique **train** resumes, so the ranking of one job's candidates doesn't depend on which other candidates happen to be in the list.

In [ ]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def tokenize(text):
    return [t for t in re.findall(r'[a-z0-9][a-z0-9+#.]*', text.lower()) if t not in ENGLISH_STOP_WORDS]

bm25_corpus = [tokenize(t) for t in train_df.resume_text.drop_duplicates()]
bm25 = BM25Okapi(bm25_corpus)  # k1=1.5, b=0.75 (library defaults)

def bm25_scores(job_text, candidates):
    q = tokenize(job_text)
    # Score candidates with the train-corpus IDF and length statistics
    scores = []
    for doc in (tokenize(c) for c in candidates):
        tf = pd.Series(doc).value_counts() if doc else pd.Series(dtype=int)
        dl = len(doc)
        s = 0.0
        for term in set(q):
            f = tf.get(term, 0)
            if f == 0:
                continue
            idf = bm25.idf.get(term, np.log((bm25.corpus_size + 0.5) / 0.5))
            s += idf * f * (bm25.k1 + 1) / (f + bm25.k1 * (1 - bm25.b + bm25.b * dl / bm25.avgdl))
        scores.append(s)
    return scores

## 6. Run all three on validation and test

In [ ]:
methods = [('Random', random_scores), ('TF-IDF', tfidf_scores), ('BM25', bm25_scores)]
summaries, per_query_all = [], []
for split_name, df in [('val', val_df), ('test', test_df)]:
    for name, fn in methods:
        s, pq = evaluate(name, fn, df, split_name)
        summaries.append(s)
        per_query_all.append(pq)

results = pd.DataFrame(summaries)
results

## 7. Is the difference real? (paired test over jobs)
A paired permutation test on NDCG@10 per job. A p-value below 0.05 means the difference is unlikely to be chance.

In [ ]:
def paired_permutation_p(a, b, n=10000, seed=SEED):
    rng = np.random.default_rng(seed)
    diff = np.asarray(a) - np.asarray(b)
    observed = abs(diff.mean())
    signs = rng.choice([-1, 1], size=(n, len(diff)))
    return float((np.abs((signs * diff).mean(axis=1)) >= observed).mean())

pq = pd.concat(per_query_all)
test_pq = pq[pq.split == 'test'].pivot(index='job_id', columns='method', values='NDCG@10')
for a, b in [('TF-IDF', 'Random'), ('BM25', 'Random'), ('BM25', 'TF-IDF')]:
    print(f'{a:6} vs {b:6}: mean diff {(test_pq[a] - test_pq[b]).mean():+.4f} | '
          f'p = {paired_permutation_p(test_pq[a], test_pq[b]):.4f}')

## 8. Save results
Per-job scores are kept so the later models (SBERT, LLM) can be tested against these baselines with the same paired test.

In [ ]:
results.to_csv(f'{RESULTS}/02_baselines_summary.csv', index=False)
pq.to_csv(f'{RESULTS}/02_baselines_per_query.csv', index=False)
print(results[results.split == 'test'][['method', 'NDCG@5', 'NDCG@10', 'P@5', 'MRR', 'ms/job (median)']].to_string(index=False))

## Done
Saved to `MyDrive/resume-job-matching/results/`:
- `02_baselines_summary.csv`: the results table rows
- `02_baselines_per_query.csv`: per-job scores for significance tests

**Next:** Notebook 3, sentence embeddings (SBERT zero-shot, then fine-tuned).